In [194]:
from neo4j import GraphDatabase
import networkx as nx
import pickle
import pandas as pd
import re
from tqdm import tqdm
import matplotlib.pyplot as plt
import math

import os
import sys
try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()
print(f'base_dir is {base_dir}')
sys.path.append(os.path.dirname(base_dir))
from util.utils import extract_hgnc_name, load_graph
from util.cmpa import compute_IF_score, run_cmpa

base_dir is /Users/yuxiaoxuan/master_thesis/KGRules/notebooks


In [73]:
# Connection to Neo4j
driver_ad = GraphDatabase.driver("bolt://localhost:7690", auth=('neo4j','test1237'))
driver_health = GraphDatabase.driver("bolt://localhost:7688", auth=('neo4j','test12345'))

In [ ]:
query_health = """
    MATCH (s)-[r:increases|decreases|regulates]->(o)
    WHERE (s:Protein OR s:Gene) AND o.id contains 'Alzheimer Disease'
    RETURN s.id AS subject, type(r) AS predicate, o.id AS object
    """
query_ad = """
    MATCH p = (s)-[*4]-(o)
    WHERE ALL(n in nodes(p) WHERE n:Protein AND n.namespace='HGNC')
    RETURN 
        [n in nodes(p) | n.id] AS node_names,
        [rel in relationships(p) | type(rel)] AS rel_types,
        length(p) AS path_length
    """

In [22]:
def extract_path_from_neo4j(driver, query):
    
    def get_triples(tx, query=query):
        result = tx.run(query)
        return [record.data() for record in result]

    with driver_noad.session() as session:
        results = session.execute_read(get_triples)
    
    df = pd.DataFrame(results)
    # Expand the lists into separate columns
    nodes_df = pd.DataFrame(df['node_names'].tolist()).add_prefix('node_')
    rels_df = pd.DataFrame(df['rel_types'].tolist()).add_prefix('rel_')

    # Combine them: Result will have columns node_0, rel_0, node_1, rel_1... node_4
    final_df = pd.concat([nodes_df, rels_df], axis=1)
    return final_df

In [24]:
df_ad = extract_path_from_neo4j(driver_ad, query_ad)

In [12]:
# Assuming 'results' is your list of records from the Neo4j driver
df = pd.DataFrame(results)

# Expand the lists into separate columns
nodes_df = pd.DataFrame(df['node_names'].tolist()).add_prefix('node_')
rels_df = pd.DataFrame(df['rel_types'].tolist()).add_prefix('rel_')

# Combine them: Result will have columns node_0, rel_0, node_1, rel_1... node_4
final_df = pd.concat([nodes_df, rels_df], axis=1)

In [13]:
df

,node_names,rel_types,path_length
0,"[g(HGNC:""BDNF""), p(HGNC:""APP"",frag(""672_713""))...","[decreases, increases, increases, increases]",4
1,"[g(HGNC:""BDNF""), p(HGNC:""APP"",frag(""672_713""))...","[decreases, increases, increases, increases]",4
2,"[g(HGNC:""BDNF""), p(HGNC:""APP"",frag(""672_713""))...","[decreases, increases, decreases, increases]",4
3,"[g(HGNC:""BDNF""), p(HGNC:""APP"",frag(""672_713""))...","[decreases, increases, decreases, increases]",4
4,"[g(HGNC:""BDNF""), p(HGNC:""APP"",frag(""672_713""))...","[decreases, increases, decreases, increases]",4
...,...,...,...
1096767,"[p(HGNC:""BMP2""), g(HGNC:""KLF10""), p(HGNC:""TGFB...","[increases, increases, increases, increases]",4
1096768,"[p(HGNC:""BMP2""), g(HGNC:""KLF10""), p(HGNC:""TGFB...","[increases, increases, increases, increases]",4
1096769,"[p(HGNC:""BMP2""), g(HGNC:""KLF10""), p(HGNC:""TGFB...","[increases, increases, increases, increases]",4
1096770,"[p(HGNC:""BMP2""), g(HGNC:""KLF10""), p(HGNC:""TGFB...","[increases, increases, increases, increases]",4


In [18]:
final_df

,node_0,node_1,node_2,node_3,node_4,rel_0,rel_1,rel_2,rel_3
0,"g(HGNC:""BDNF"")","p(HGNC:""APP"",frag(""672_713""))","p(HGNC:""BDNF"")","g(HGNC:""BDNF"")","g(HGNC:""SORL1"")",decreases,increases,increases,increases
1,"g(HGNC:""BDNF"")","p(HGNC:""APP"",frag(""672_713""))","p(HGNC:""BDNF"")","p(HGNC:""CD274"")","p(HGNC:""IL10"")",decreases,increases,increases,increases
2,"g(HGNC:""BDNF"")","p(HGNC:""APP"",frag(""672_713""))","p(HGNC:""BDNF"")","p(HGNC:""APP"")","p(HGNC:""KAT5"",pmod(Ph))",decreases,increases,decreases,increases
3,"g(HGNC:""BDNF"")","p(HGNC:""APP"",frag(""672_713""))","p(HGNC:""BDNF"")","p(HGNC:""APP"")","p(HGNC:""GNAI1"")",decreases,increases,decreases,increases
4,"g(HGNC:""BDNF"")","p(HGNC:""APP"",frag(""672_713""))","p(HGNC:""BDNF"")","p(HGNC:""APP"")","p(HGNC:""HINFP"",var(""A,K,5""))",decreases,increases,decreases,increases
...,...,...,...,...,...,...,...,...,...
1096767,"p(HGNC:""BMP2"")","g(HGNC:""KLF10"")","p(HGNC:""TGFB3"")","p(HGNC:""IL34"")","p(HGNC:""CSF1R"")",increases,increases,increases,increases
1096768,"p(HGNC:""BMP2"")","g(HGNC:""KLF10"")","p(HGNC:""TGFB3"")","p(HGNC:""IL34"")","p(FPLX:""TGFB"")",increases,increases,increases,increases
1096769,"p(HGNC:""BMP2"")","g(HGNC:""KLF10"")","p(HGNC:""TGFB3"")","p(HGNC:""IL34"")","p(HGNC:""HMOX1"")",increases,increases,increases,increases
1096770,"p(HGNC:""BMP2"")","g(HGNC:""KLF10"")","p(HGNC:""TGFB3"")","p(HGNC:""IL34"")","p(HGNC:""IDE"")",increases,increases,increases,increases


## Subgraph -> KG-Rules

Idea:
+ query Neo4j
+ extract subgraphs (clusters)
+ calculate scores of each cluster for each patient, serve as feature scores

In [254]:
def extract_subgraphs_from_neo4j(driver, query=None):
    if not query:
        query = """
        MATCH p = (s)-[:increases|decreases*..4]-(o)
        WHERE ALL(n in nodes(p) WHERE n:Protein OR n:Gene)
        UNWIND nodes(p) AS n
        UNWIND relationships(p) AS r
        WITH collect(DISTINCT n) AS nodes, collect(DISTINCT r) AS rels
        RETURN nodes, rels
        """
    def query_neo4j(tx):
        result = tx.run(query)
        return result.single()
    with driver.session() as session:
        record = session.execute_read(query_neo4j)
    
    # Extract nodes with their properties
    nodes = [{'name':dict(n)['id'],"properties": dict(n)} for n in record["nodes"]]
    
    # Extract relationships with their properties
    edges = [{
        "start": dict(r.start_node)['id'], 
        "end": dict(r.end_node)['id'], 
        "type": r.type, 
        "properties": dict(r)
    } for r in record["rels"]]
    
    return {"nodes": nodes, "edges": edges}

subgraph_ad = extract_subgraphs_from_neo4j(driver_ad)

In [255]:
import ast

def get_ad_subgraph_names(subgraph_ad:dict):
    subgraph_names = []
    for edge_dict in subgraph_ad['edges']:
        subg = edge_dict.get('properties',{}).get('subgraph','')
        #print(type(subg))
        if isinstance(subg, str) and subg.strip():
            try:
                subg_list = ast.literal_eval(subg)
                subgraph_names.extend(subg_list)
            except:
                print(subg)
    subgraph_names = set(subgraph_names)
    print('The number of subgraphs:',len(subgraph_names))
    
    return subgraph_names


### Extract subgraphs based on edge.properties.subgraph
+ the curated KG edges have properties 'subgraph', so I will extract clusters of each subgraph

In [256]:
def get_subgraph_query(subgraph_name):
    query = f"""
        MATCH p = (s)-[rels_in_path:increases|decreases|regulates*..4]-(o)
        WHERE ALL(n in nodes(p) WHERE n:Protein OR n:Gene) 
            AND ALL(rel in rels_in_path WHERE rel.subgraph CONTAINS '{subgraph_name}')
        
        // Use unique names for the UNWIND variables
        UNWIND nodes(p) AS individual_node
        UNWIND relationships(p) AS individual_rel
        
        WITH collect(DISTINCT individual_node) AS nodes, 
            collect(DISTINCT individual_rel) AS rels
        RETURN nodes, rels
    """
    return query

def extract_ad_subgraphs(driver, subgraph_names:list):
    subgraphs_dict = {}
    for subgraph_name in subgraph_names:
        query = get_subgraph_query(subgraph_name)
        #print(query)

        # extract subgraph and store in a dict
        subg = extract_subgraphs_from_neo4j(driver, query)
        print(f"{subgraph_name} has {len(subg['nodes'])} nodes and {len(subg['edges'])} edges")
        subg_name = '_'.join(subgraph_name.split(' '))
        subgraphs_dict[subg_name]=subg
        #break
    return subgraphs_dict

### CMPA-Score as KG-Features
+ for each subgraph_pathway:
  + for each component graph:
    + calculate $CMPA$ score for eahc hub node
    + use the average $CMPA$ as each patient's subgraph-feature

Convert subgraphs_dict to a dictionary of nx graphs

In [257]:
def convert_subgraphs_to_nx(subgraphs_dict):
    confidence_map = {'Very High':1, 'High': 0.8, 'Medium': 0.6, 'Low':0.2}
    subG_dict = {}
    for subg_name, subg in subgraphs_dict.items():
        print(f'\nPathway name: {subg_name}')
        
        # create a subgraph for each pathway 
        sub_G = nx.MultiDiGraph()
        for node in subg['nodes']:
            #print(node)
            id = node.get('name','')
            label = node.get('properties',{}).get('type',"")
            #print(label)
            protein = node.get('properties',{}).get('name',"")
            properties = node.get('properties')
            #properties.pop('label', None)
            sub_G.add_node(id, labels=label, protein=protein, properties=properties)
        # add edges
        print(f"Added {len(sub_G.nodes)} nodes to subGraph")
        for edge in subg['edges']:
            #print(edge)
            src = edge['start']
            dst = edge['end']
            rel = edge['type']
            confidence = edge.get('properties',{}).get('confidence', 0)
            
            props = edge.get('properties', {}).copy()
            # Remove 'confidence' from props so it's not passed twice
            props.pop('confidence', None)
            if isinstance(confidence,str):
                conf = ast.literal_eval(confidence)[0]
                confidence = confidence_map[conf]
            else:
                confidence = float(confidence)
            #print(confidence)
            sub_G.add_edge(src, dst, type=rel, confidence=confidence, **props)
            #break
        #print(f"Added {len(sub_G.edges)} edges to subGraph")

        subG_dict[subg_name] = sub_G
    
    return subG_dict

#G_subgraphs = convert_subgraphs_to_nx(subgraphs_dict)

In [172]:
for _, subG in G_subgraphs.items():
    for node in subG.nodes(data=True):
        print(node)
        break
    for edge in subG.edges(data=True, keys=True):
        print(edge)
        break

    break

('p(HGNC:"MAPT",pmod(Ph))', {'labels': 'Protein', 'protein': 'MAPT', 'properties': {'involved_genes': 'MAPT', 'species': 9606.0, 'namespace': 'HGNC', 'name': 'MAPT', 'id': 'p(HGNC:"MAPT",pmod(Ph))', 'type': 'Protein'}, 'IF_score': 0.0})
('p(HGNC:"SNCA")', 'p(HGNC:"MAPT",pmod(Ph))', 0, {'type': 'increases', 'confidence': 0.8, 'evidence': '["In this in vitro study, we found that: (a) alpha-SN directly stimulates the phosphorylation of tau by glycogen synthase kinase-3beta (GSK-3beta), (b) alpha-SN forms a heterotrimeric complex with tau and GSK-3beta, and (c) the nonamyloid beta component (NAC) domain and an acidic region of alpha-SN are responsible for the stimulation of GSK-3beta-mediated tau phosphorylation. Thus, it is concluded that alpha-SN functions as a connecting mediator for tau and GSK-3beta, resulting in GSK-3beta-mediated tau phosphorylation. Because the expression of alpha-SN is promoted by oxidative stress, the accumulation of alpha-SN induced by such stress may directly i

+ CMPA- not compatible with cylic graphs

In [258]:
# CMPA Alogorithm -- Not compatible with cyclic graphs
def compute_IF_score(graph, node):
    IF_score = graph.nodes[node]['IF_score']
    
    for pred in graph.predecessors(node):
        edge_dict = graph[pred][node]
        #print(edge_dict)
        rel_type = graph[pred][node][0]['type']
        if 'increases' in rel_type.lower():
            print(pred, "-->", node)
            print("Before: ", graph.nodes[node]["IF_score"])
            
            IF_score += graph.nodes[pred]["IF_score"] * 1.0
            
            print("After: ", IF_score)
        elif 'decreases' in rel_type.lower():
            print(pred, "--|", node)
            print("Before: ", graph.nodes[node]["IF_score"])
            
            IF_score += graph.nodes[pred]["IF_score"] * -1.0
            
            print("After: ", IF_score)
        else:
            IF_score += graph.nodes[pred]["IF_score"] * 0.1
    
    return IF_score

def run_cmpa(data, graph):
    """Runs the CMPA algorithm on a networkx graph (Handling cyclic graphs).
    Arguments:
    - data: a dictionary with gene_symbol: log_FC score
    - graph: a networkx graph with nodes and edges annotated with 'labels' and 'properties' fields.
    Returns:
    - a pandas DataFrame with all the hubs and their scores."""
    # 1. Check for cycles
    if not nx.is_directed_acyclic_graph(graph):
        print("Warning: Graph contains cycles! CMPA logic may need iteration.")

    result_list = []
    hub_list = []
    
    for node,attr in graph.nodes(data=True):
        node_protein = attr.get('protein')
        if node_protein in data.keys():
            graph.nodes[node]['IF_score'] = data[node_protein]
        else:
            graph.nodes[node]['IF_score'] = 0.0
            
        if any(graph.predecessors(node)):
            hub_list.append(node)
        
        # graph.nodes[node]['IF_score'] = graph.nodes[node]['weight']
    
    print("Hub List: ", hub_list)
    
    while hub_list:
        for node in graph.nodes:
            if any([n in hub_list for n in graph.predecessors(node)]):
                continue
            if node not in hub_list:
                continue
            print("--------------------")
            print("Processing node: ", node)
            print("Current IF Score: ", graph.nodes[node]['IF_score'])
            
            graph.nodes[node]['IF_score'] = compute_IF_score(graph, node)
            
            print("New IF Score: ", graph.nodes[node]['IF_score'])
            
            hub_list.remove(node)
    
    for node in graph.nodes:
        result_list.append({'hub': node, 'score': graph.nodes[node]['IF_score']})

    result = pd.DataFrame(result_list)
    
    return result

def plot_subgraph(G, node_label=False, edge_label=False, title="Biological Subgraph"):
    """
    Plots a NetworkX graph with color-coding for Protein/Gene nodes 
    and labels for relationship types.
    """
    plt.figure(figsize=(10, 6))
    
    # 1. Define Layout (Force-directed)
    # k regulates the distance between nodes
    pos = nx.spring_layout(G, k=0.5, seed=42)
    
    # 2. Color-code nodes based on their labels
    # We assume 'labels' was stored as a list in the node properties
    node_colors = []
    for node_id, data in G.nodes(data=True):
        label = data.get('labels', '')
        if 'Protein' in label.capitalize():
            node_colors.append('salmon')
        elif 'Gene' in label:
            node_colors.append('skyblue')
        else:
            node_colors.append('lightgrey')
    edge_colors = []
    for _,_,edtp in G.edges(data=True):
        edge_type = edtp.get('type','')
        if 'increase' in edge_type:
            edge_colors.append('black')
        elif 'decrease' in edge_type:
            edge_colors.append('blue')
        else:
            edge_colors.append('gray')
    # 3. Draw Nodes and Edges
    dynamic_size = max(10, 1000 / (len(G.nodes)**0.8))
    nx.draw_networkx_nodes(G, pos, node_size=dynamic_size, node_color=node_colors, alpha=0.9)
    nx.draw_networkx_edges(G, pos, width=1.5, alpha=0.5, edge_color=edge_colors)
    if node_label:
        # 4. Add Labels
        # Use 'name' property for nodes
        node_labels = {n:n for n in G.nodes}
        nx.draw_networkx_labels(G, pos, labels=node_labels, font_size=10, font_family="sans-serif")
    if edge_label:
        # Use 'type' property for edges (e.g., increases, decreases)
        edge_labels = {(u, v): d.get('type', '') for u, v, d in G.edges(data=True)}
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)

    plt.title(title)
    plt.axis('off')
    plt.show()
    

### CMPA Algorithm - Cyclic Graph Compatible
+ Iterative convergence is applied on cycle-nodes by introducing a damping ($<1$) in computing new $IF$
+ $compute\_IF\_score$ always starts with the $base\_score$
+ $nx.topological\_sort$ decides the list of nodes to iterate in downstreat -> upstream order, such that each node always compute $IF$ from freshly processed predecedors
+ $nx.condensation(graph)$ breaks a cyclic-graph to a DAG => {ssc_idx: {nodes in SSC}, ssc_idx: node_not_in_ssc}
+ SSC = Strongly Connected Component (cycle)

In [243]:
# CMPA Algorithm - Cyclic Graph Compatible
def compute_IF_score(graph, node):
    IF_score = graph.nodes[node]['base_score']
    
    for pred in graph.predecessors(node):
        edge_dict = graph[pred][node]
        #print(edge_dict)
        rel_type = graph[pred][node][0]['type']
        if 'increases' in rel_type.lower():
            # print(pred, "-->", node)
            # print("Before: ", graph.nodes[node]["IF_score"])
            
            IF_score += graph.nodes[pred]["IF_score"] * 1.0
            
            # print("After: ", IF_score)
        elif 'decreases' in rel_type.lower():
            # print(pred, "--|", node)
            # print("Before: ", graph.nodes[node]["IF_score"])
            
            IF_score += graph.nodes[pred]["IF_score"] * -1.0
            
            # print("After: ", IF_score)
        else:
            IF_score += graph.nodes[pred]["IF_score"] * 0.1
    
    return IF_score

def get_max_iter(graph):
    # Diameter of the underlying undirected graph ≈ longest propagation path
    try:
        diameter = nx.diameter(graph.to_undirected())
    except nx.NetworkXError:
        # Disconnected graph — use largest component's diameter
        largest = max(nx.connected_components(graph.to_undirected()), key=len)
        subgraph = graph.subgraph(largest).to_undirected()
        diameter = nx.diameter(subgraph)
    
    # 3× diameter is generous; log factor adds safety for dense cycle interactions
    return max(10, int(3 * diameter * math.log1p(graph.number_of_nodes())))

def run_cmpa(data, graph,tol=1e-6, damping=0.85):
    """
    Runs the CMPA algorithm on a networkx graph, compatible with cyclic graphs.
    
    Uses iterative convergence: repeatedly updates all node scores until they
    stabilize (delta < tol) or max_iter is reached. Cyclic nodes will reach
    a fixed-point equilibrium rather than causing an infinite loop.

    Arguments:
    - data:     dict of {gene_symbol: log_FC score}
    - graph:    networkx MultiDiGraph with 'protein' on nodes, 'type' on edges
    - tol:      convergence tolerance — stops when max score delta < tol (default: 1e-6)
    - damping:  damping factor for cyclic nodes to prevent score explosion (default: 0.85)

    Returns:
    - pandas DataFrame with columns ['hub', 'score']
    """
    is_dag = nx.is_directed_acyclic_graph(graph)
    if not is_dag:
        cycles = list(nx.simple_cycles(graph))
        print(f"Warning: Graph contains {len(cycles)} cycle(s). Using iterative convergence.")
        print(f"  Cycles detected: {cycles}")
    else:
        print("Graph is a DAG. Running standard topological CMPA.")

    # ── Step 1: Initialise scores ──────────────────────────────────────────────
    for node, attr in graph.nodes(data=True):
        node_protein = attr.get('protein')
        base = data[node_protein] if node_protein in data else 1.0
        graph.nodes[node]['base_score'] = base   # immutable seed value
        graph.nodes[node]['IF_score']   = base   # will be updated each iteration

    # ── Step 2: Determine processing order ────────────────────────────────────
    if is_dag:
        # Topological order guarantees each predecessor is processed before its successor
        ordered_nodes = list(nx.topological_sort(graph))
    else:
        # For cyclic graphs: isolate cycle nodes so we can apply damping to them
        cycle_nodes = set(n for cycle in nx.simple_cycles(graph) for n in cycle)
        # Topologically sort the condensation (SCC DAG), then expand each SCC's members
        # SCC = Strongly Connected Components
        condensation  = nx.condensation(graph)
        ordered_nodes = [
            node
            for scc_idx in nx.topological_sort(condensation)
            for node in condensation.nodes[scc_idx]['members']   # O(1) lookup per SCC
        ]

    # ── Step 3: Iterative convergence ─────────────────────────────────────────
    max_iter = get_max_iter(graph)
    print('Max iteration:', max_iter)
    for iteration in tqdm(range(max_iter), desc="Iterative convergence"):
        max_delta = 0.0

        for node in ordered_nodes:
            old_score = graph.nodes[node]['IF_score']
            new_score = compute_IF_score(graph, node)

            # Apply damping only to nodes that are part of a cycle
            if not is_dag and node in cycle_nodes:
                new_score = damping * new_score + (1 - damping) * graph.nodes[node]['base_score']

            graph.nodes[node]['IF_score'] = new_score
            max_delta = max(max_delta, abs(new_score - old_score))

        #print(f"Iteration {iteration + 1:>3}: max_delta = {max_delta:.2e}")

        # Early exit once scores have converged
        if max_delta < tol:
            print(f"Converged after {iteration + 1} iteration(s).")
            break
    else:
        print(f"Warning: Reached max_iter={max_iter} without full convergence "
              f"(final max_delta={max_delta:.2e}). Scores are approximate.")

    # ── Step 4: Collect results ────────────────────────────────────────────────
    result_list = [
        {'hub': node, 'score': graph.nodes[node]['IF_score']}
        for node in graph.nodes
    ]
    return pd.DataFrame(result_list)

some CMPA modifications doesn't work well

In [ ]:
# scale the output of compute_IF_score doesn't work well
def compute_IF_score(graph, node, damping=0.85):
    predecessors = list(graph.predecessors(node))
    if not predecessors:
        return graph.nodes[node]['base_score']

    influence = 0.0
    for pred in predecessors:
        rel_type = graph[pred][node][0]['type']
        if 'increases' in rel_type.lower():
            weight = 1.0
        elif 'decreases' in rel_type.lower():
            weight = -1.0
        else:
            weight = 0.1
        # Use running IF_score, not base_score — this is where hub accumulation happens
        influence += graph.nodes[pred]['IF_score'] * weight

    # No division — hub nodes with many predecessors accumulate more signal
    # damping prevents explosion: 85% predecessor signal, 15% own base anchors it
    return damping * influence + (1 - damping) * graph.nodes[node]['base_score']

def get_max_iter(graph):
    # Diameter of the underlying undirected graph ≈ longest propagation path
    try:
        diameter = nx.diameter(graph.to_undirected())
    except nx.NetworkXError:
        # Disconnected graph — use largest component's diameter
        largest = max(nx.connected_components(graph.to_undirected()), key=len)
        subgraph = graph.subgraph(largest).to_undirected()
        diameter = nx.diameter(subgraph)
    
    # 3× diameter is generous; log factor adds safety for dense cycle interactions
    return max(10, int(3 * diameter * math.log1p(graph.number_of_nodes())))

def run_cmpa(data, graph,tol=1e-6, damping=0.85):
    """
    Runs the CMPA algorithm on a networkx graph, compatible with cyclic graphs.
    
    Uses iterative convergence: repeatedly updates all node scores until they
    stabilize (delta < tol) or max_iter is reached. Cyclic nodes will reach
    a fixed-point equilibrium rather than causing an infinite loop.

    Arguments:
    - data:     dict of {gene_symbol: log_FC score}
    - graph:    networkx MultiDiGraph with 'protein' on nodes, 'type' on edges
    - tol:      convergence tolerance — stops when max score delta < tol (default: 1e-6)
    - damping:  damping factor for cyclic nodes to prevent score explosion (default: 0.85)

    Returns:
    - pandas DataFrame with columns ['hub', 'score']
    """
    is_dag = nx.is_directed_acyclic_graph(graph)
    if not is_dag:
        cycles = list(nx.simple_cycles(graph))
        print(f"Warning: Graph contains {len(cycles)} cycle(s). Using iterative convergence.")
        print(f"  Cycles detected: {cycles}")
    else:
        print("Graph is a DAG. Running standard topological CMPA.")

    # ── Step 1: Initialise scores ──────────────────────────────────────────────
    for node, attr in graph.nodes(data=True):
        node_protein = attr.get('protein')
        base = data[node_protein] if node_protein in data else 0.0
        graph.nodes[node]['base_score'] = base   # immutable seed value
        graph.nodes[node]['IF_score']   = base   # will be updated each iteration

    # ── Step 2: Determine processing order ────────────────────────────────────
    if is_dag:
        # Topological order guarantees each predecessor is processed before its successor
        ordered_nodes = list(nx.topological_sort(graph))
    else:
        # For cyclic graphs: isolate cycle nodes so we can apply damping to them
        cycle_nodes = set(n for cycle in nx.simple_cycles(graph) for n in cycle)
        # Topologically sort the condensation (SCC DAG), then expand each SCC's members
        # SCC = Strongly Connected Components
        condensation  = nx.condensation(graph)
        ordered_nodes = [
            node
            for scc_idx in nx.topological_sort(condensation)
            for node in condensation.nodes[scc_idx]['members']   # O(1) lookup per SCC
        ]

    # ── Step 3: Iterative convergence ─────────────────────────────────────────
    if is_dag:
        # Single pass in topological order — exact, no damping needed
        for node in ordered_nodes:
            if list(graph.predecessors(node)):   # only update non-source nodes
                graph.nodes[node]['IF_score'] = compute_IF_score(graph, node, damping=1.0)
                # damping=1.0 means pure predecessor signal, no base_score blending
    else:
        # Cyclic graph — iterate until convergence
        
        max_iter = get_max_iter(graph)
        print('Max iteration:', max_iter)
        for iteration in tqdm(range(max_iter), desc="Iterative convergence"):
            max_delta = 0.0

            for node in ordered_nodes:
                old_score = graph.nodes[node]['IF_score']
                new_score = compute_IF_score(graph, node)

                # Apply damping only to nodes that are part of a cycle
                if not is_dag and node in cycle_nodes:
                    new_score = damping * new_score + (1 - damping) * graph.nodes[node]['base_score']

                graph.nodes[node]['IF_score'] = new_score
                max_delta = max(max_delta, abs(new_score - old_score))

            #print(f"Iteration {iteration + 1:>3}: max_delta = {max_delta:.2e}")

            # Early exit once scores have converged
            if max_delta < tol:
                print(f"Converged after {iteration + 1} iteration(s).")
                break
        else:
            print(f"Warning: Reached max_iter={max_iter} without full convergence "
                f"(final max_delta={max_delta:.2e}). Scores are approximate.")

    # ── Step 4: Collect results ────────────────────────────────────────────────
    result_list = [
        {'hub': node, 'score': graph.nodes[node]['IF_score']}
        for node in graph.nodes
    ]
    return pd.DataFrame(result_list)


+ run cmpa and calculate the subgraph-features for patients

In [ ]:
def ad_subgraph_to_rules(
                         output_dir:str,
                         expression_path:str="../Anyburl/data/adni_gene_cleaned.csv",
                         uri:str="bolt://localhost:7690", 
                         username:str="neo4j", 
                         passwaord:str="test1237", 
                         query:str|None=None,
                         G_subgraphs:dict|None = None):
                         
    driver = GraphDatabase.driver(uri, auth=(username, passwaord))
    if not query:
        query = """
        MATCH p = (s)-[:increases|decreases*..4]-(o)
        WHERE ALL(n in nodes(p) WHERE n:Protein OR n:Gene)
        UNWIND nodes(p) AS n
        UNWIND relationships(p) AS r
        WITH collect(DISTINCT n) AS nodes, collect(DISTINCT r) AS rels
        RETURN nodes, rels
        """
    if not G_subgraphs:
        # 1. Query Neo4j to Get subgraph names and then extract according subgraphs
        subgraph_flat = extract_subgraphs_from_neo4j(driver, query)
        subgraph_names = get_ad_subgraph_names(subgraph_ad=subgraph_flat)
        subgraphs_dict = extract_ad_subgraphs(driver, subgraph_names)
        # convert graph_dictionary to nx.graph
        G_subgraphs = convert_adsubgraphs_to_nx(subgraphs_dict=subgraphs_dict)

    # 2. Compute CMPA for all subgraphs
    exp_df = pd.read_csv(expression_path, index_col=0).T
    df_patient_cmpa = pd.DataFrame(index=exp_df.index, columns = list(G_subgraphs.keys()))
    
    for i in tqdm(range(len(exp_df)), desc="Computing Subgraph CMPA Score For Patients"):
        patient_data = exp_df.iloc[i].to_dict()

        df_subgraphs = {}
        subgraph_scores = []
        for subg_name, subG in G_subgraphs.items():
            #plot_subgraph(subG, True, True, subg_name)
            print('='*60)
            print(f"Run CMPA on {subg_name}")
            df_subG = run_cmpa(patient_data, subG)
            df_subgraphs[subg_name] = df_subG
            
            subgraph_scores.append(df_subG['score'].sum())
        df_patient_cmpa.iloc[i] = subgraph_scores
    
    # save features file
    save_path = os.path.join(output_dir, 'ad_subgraph_features.csv')
    df_patient_cmpa.to_csv(save_path, index=False)
    return df_patient_cmpa


### Finding connected-components as clusters